# 🔬 IMD Streaming Workshop - Solutions

Complete solutions for the workshop exercises with live visualizations.

---

## Exercise 1 Solution: Charge Density Analysis

Calculate the total charge within 4 Å of the protein at each timestep.

In [ ]:
from imdclient.IMD import IMDReader
import MDAnalysis as mda
import numpy as np
from graph_utils import live_plot

u = mda.Universe("sample_simulation/GROMACS/input/start.gro", "imd://localhost:8889", buffer_size=100*1024**2)

try:
    # Select solvent atoms within 4 Å of protein (updating each frame)
    nearby = u.select_atoms('not protein and around 4 protein', updating=True)

    # Create live plot
    plot = live_plot(
        title="Charge Density within 4 Å",
        ylabel="Total Charge (e)",
        update_interval=1
    )

    for ts in u.trajectory:
        # Calculate total charge in selection
        total_charge = np.sum(nearby.charges)
        plot['update'](ts.time, total_charge)
        
        
        print(f"Frame {ts.frame:4d} | Time: {ts.time:8.2f} ps | Charge: {total_charge:+.4f} e | Atoms: {nearby.n_atoms}")

except Exception as e:
    print(f"\n\nError: {e}")
finally:
    u.trajectory.close()
    plot['close']()

---

## Exercise 2 Solution: Backbone RMSD

Calculate the backbone RMSD relative to the starting structure.

**What is backbone?** Only the main chain atoms (N, CA, C, O) that form the protein scaffold, excluding side chains.

**Why backbone RMSD?** More stable than all-atom RMSD since side chains are flexible. Captures overall structural changes.

In [ ]:
from imdclient.IMD import IMDReader
import MDAnalysis as mda
from MDAnalysis.analysis import rms
import numpy as np
from graph_utils import live_plot

u = mda.Universe("sample_simulation/GROMACS/input/start.gro", "imd://localhost:8889", buffer_size=100*1024**2)

try:
    # Select backbone atoms (N, CA, C, O - the protein main chain)
    backbone = u.select_atoms("protein and backbone")
    
    # Variable to store reference structure
    reference_positions = None

    # Create live plot
    plot = live_plot(
        title="Backbone RMSD from Starting Structure",
        ylabel="RMSD (Å)",
        update_interval=1
    )

    for ts in u.trajectory:
        # Save reference positions from first frame
        if ts.frame == 0:
            reference_positions = backbone.positions.copy()
            continue  # Skip RMSD calculation for frame 0 (RMSD = 0)
        
        # Calculate RMSD with alignment (superposition=True)
        rmsd_value = rms.rmsd(
            backbone.positions,
            reference_positions,
            superposition=True  # Align structures before calculating RMSD
        )
        
        plot['update'](ts.time, rmsd_value)

except Exception as e:
    print(f"\n\nError: {e}")
finally:
    u.trajectory.close()
    plot['close']()